In [1]:
import pandas as pd
import numpy as np
import robustsharpe as rs

In [2]:
# average_model_path = "../result/grpo/03_total_avg.csv"
# equal_strategy_path = "../result/benchmark/0.003_benchmark.csv"
average_model_path = "../result/grpo_del/0.003_total_del_avg.csv"
equal_strategy_path = "../result/benchmark_del/0.003_benchmark.csv"

In [3]:
grpo_best = pd.read_csv(average_model_path, index_col=0)

In [4]:
grpo_best

,GRPO,PPO,SAC
2019-01-30,0.021799,-0.016721,-0.014740
2019-02-28,0.020955,0.020955,0.019843
2019-03-28,-0.007319,-0.005156,-0.008855
2019-04-26,0.023269,0.027753,0.019544
2019-05-24,-0.025046,-0.027919,-0.027554
...,...,...,...
2024-09-20,0.007695,-0.014056,0.003555
2024-10-18,0.026079,0.012836,0.013755
2024-11-15,-0.006021,-0.003139,-0.015770
2024-12-16,0.065300,0.058342,0.053909


In [5]:
eqaul = pd.read_csv(equal_strategy_path, index_col=0)

In [6]:
eqaul.columns

Index(['risk_parity', 'min_var', 'max_sharpe', 'paa', 'equal_strategy',
       'equal_asset_weight'],
      dtype='object')

In [87]:
my_strategy_returns = np.array(grpo_best["SAC"])

In [88]:
# my_strategy_returns = np.array(eqaul["equal_asset_weight"])

In [89]:
eqaul.columns

Index(['risk_parity', 'min_var', 'max_sharpe', 'paa', 'equal_strategy',
       'equal_asset_weight'],
      dtype='object')

In [90]:
# benchmark_returns = np.array(grpo_best["PPO"])
benchmark_returns = np.array(eqaul["min_var"])

In [91]:
returns = np.stack([my_strategy_returns, benchmark_returns], axis=1)  # shape: (T, 2)

In [92]:
# returns = np.stack([benchmark_returns, my_strategy_returns], axis=1)  # shape: (T, 2)

In [93]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,         # shape (T, 2)
#     b_vec=[1, 2, 4, 6],  # 후보 block size
#     alpha=0.05,
#     M=199,                    # bootstrap per test
#     K=1000,                    # pseudo-sequence 생성 횟수
#     T_start= 20
# )


In [94]:
b_vec=[1, 2, 3, 4, 5, 6]

In [95]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,
#     b_vec=[1, 2, 3, 4, 5, 6],  # T=76이므로 6 이상은 피하는게 좋음
#     alpha=0.05,
#     K=300,
#     M=99,
#     T_start=20
# )


In [96]:
SRs, diff, ci, pval, se, d = rs.bootstrap_inference(
    returns=returns,
    block_size=4,     # 논문 추천값 (T=120 기준)
    alpha=0.05,
    M=1000
)

print("Sharpe ratio (벤치마크, 내):", SRs)
print("Sharpe ratio 차이:", diff)
print("95% 신뢰구간:", ci)
print("t-통계량", d)
print("p-value:", pval)

Sharpe ratio (벤치마크, 내): [0.19507363 0.31417556]
Sharpe ratio 차이: 0.1191019241410429
95% 신뢰구간: (np.float64(-0.10876401144100126), np.float64(0.3469678597230871))
t-통계량 1.0875814373685704
p-value: 0.3046953046953047
